# Experiment: does prediction help when a simple latent is seen through many redundant, noisy channels?

**Hypothesis.** Predictive self-supervision learns features that are sensitive to changes in the dynamics and relatively insensitive to how those dynamics are observed. Its advantage should grow when a low-dimensional state with simple dynamics is observed through many redundant, individually noisy channels: per-channel noise is unpredictable, so a latent predictor can ignore it, while a reconstruction objective must spend capacity on every channel and window statistics face the full dimension.

**Data (a 2 × 2 design, sampled independently).**
- *Dynamics* (`main`, the target regime, K = 2, dwell 200 samples): a 2-D state $s_{t+1} = \alpha R(\pm\omega) s_t + \epsilon_t$, $\alpha = 0.98$, $\omega = 0.12$ rad/sample, process noise $(1-\alpha^2) I$. Both regimes have the same stationary law $\mathcal N(0, I)$ and differ only in the direction of rotation (`oscillator`, `quadrature=True`, `freq=±ω/2π`).
- *Observation*: a fixed nonlinear rendering into D channels (`render`; default random Fourier features $\sqrt2\cos(a_j\cdot s + b_j)$, injective, not invertible by any linear map; `kind="bumps"` renders moving blobs on a pixel strip instead).
- *Sensor conditions* (`sensor`, K = 2, dwell 600, independent of the dynamics): per-channel gain and bias (`affine`) and pixel noise σ = 0.5 or 1.5 (`noise`) switch with this process.

Regimes are defined by the mechanism (rotation direction), never by the sensor condition.

**Measurements.**
1. Regime recovery as in the other experiments: every feature learner × every detector, sample-level clustering accuracy and change-point metrics. Detectors include the switching-dynamics probe (AR-HMM) alongside static clustering.
2. *Sensitivity* (`cpbench/eval/invariance.py`): counterfactual pairs that share the hidden trajectory and every noise draw. Each score is the paired distance in the representation divided by the distance between unrelated windows (0: invariant, ~1: as different as unrelated windows):
   - **sensor**: same trajectory, other sensor condition;
   - **dynamics**: same innovations, other rotation direction, same sensor condition.

   A representation suited to regime discovery scores high on dynamics and low on sensor.
3. *Decodability* (diagnostic, uses labels): linear-probe accuracy of the regime and of the sensor condition from the test features.

**The dimension sweep.** Set D with `CPBENCH_D` (default 64) and the rendering with `CPBENCH_RENDER` (default `fourier`). The full sweep (D ∈ {8, 64, 256, 1024} × several seeds) is an HPC job: `python -m cpbench.hpc.make_sweep cpbench/configs/sweep_highdim.yaml` trains into the same run directory, and re-running this notebook per D then only scores. The last section compares every D found on disk.

**Caveats.** Rotation direction is visible to a second-order statistic of the clean state (the antisymmetric lagged cross-covariance flips sign), so the rendering and the noise are what make it hard. `oracle_clean` applies that statistic to the clean 2-D state: a ceiling. For D > 128 the raw features with D² entries (`raw_cov`, `raw_coskew`, `raw_lagcov`, `raw_bispec`) are skipped (memory).

In [ ]:
import os, glob, re, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from cpbench import Benchmark, make_learner, viz
from cpbench.data import Dataset
from cpbench.features import LEARNERS
from cpbench.detectors import DETECTORS
from cpbench.eval.invariance import sensitivity, decodability
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "font.size": 9})
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 80)
warnings.filterwarnings("ignore", message="FastICA did not converge")

QUICK = os.environ.get("CPBENCH_QUICK") == "1"     # ✏️ True: plumbing check only (20 steps, tiny splits)
D = int(os.environ.get("CPBENCH_D", 64))           # ✏️ observation dimension
KIND = os.environ.get("CPBENCH_RENDER", "fourier") # ✏️ "fourier" or "bumps"
RUNS = "cpbench_runs_highdim" + ("_quick" if QUICK else "")
TAG = f"D{D}_{KIND}" + ("_quick" if QUICK else "")
OUT, OUT_INV = f"cpbench_results/highdim_{TAG}.csv", f"cpbench_results/highdim_{TAG}_invariance.csv"
print("QUICK" if QUICK else "full run", TAG, "->", RUNS)

## 1. Dataset

In [ ]:
# ✏️ the experiment's dataset
OMEGA, ALPHA = 0.12, 0.98

def highdim_dataset(D, kind="fourier", quick=False):
    return dict(
        regimes=dict(main=dict(type="markov", K=2, dwell=200), sensor=dict(type="markov", K=2, dwell=600)), target="main",
        sources=[dict(type="oscillator", d=1, quadrature=True, freq=[OMEGA / (2 * np.pi), -OMEGA / (2 * np.pi)], damp=ALPHA)],
        transforms=[dict(type="render", D=D, kind=kind),
                    dict(type="affine", gain=dict(base=1.0, spread=0.3), bias=dict(base=0.0, spread=0.2), regime="sensor"),
                    dict(type="noise", sigma=[0.5, 1.5], regime="sensor")],
        seed=0,
        splits=dict(test=[2, 3000], val=[2, 3000]) if quick else dict(test=[8, 16000], val=[4, 8000]))

DATASET = highdim_dataset(D, KIND, QUICK)
W, GAP = 32, 32                         # ✏️ window and context-target gap (samples); a rotation period is ~52 samples
ds = Dataset(DATASET)
data = ds.test_set()
print(ds.info.id)
print(f"K={ds.K}, D={ds.D}, test split {data['obs'].shape}; sensor condition on {data['S_sensor'].mean():.0%} of samples")

In [ ]:
# one series: the clean 2-D state, the dynamics regime, the sensor condition and the observed channels
n = 1500
fig, ax = plt.subplots(3, 1, figsize=(11, 5.5), sharex=True, gridspec_kw=dict(height_ratios=[1.3, 0.4, 2.2]))
ax[0].plot(data["clean"][0, :n]); ax[0].set_ylabel("state s")
ax[1].imshow(np.stack([data["S_s"][0, :n], data["S_sensor"][0, :n]]), aspect="auto", cmap="tab10", vmin=0, vmax=9, interpolation="nearest")
ax[1].set_yticks([0, 1], ["dynamics", "sensor"])
show = min(D, 64)
ax[2].imshow(data["obs"][0, :n, :show].T, aspect="auto", cmap="RdBu_r", vmin=-3, vmax=3, interpolation="nearest")
ax[2].set_ylabel(f"channels (first {show} of {D})"); ax[2].set_xlabel("sample"); plt.tight_layout(); plt.show()

In [ ]:
# check: the marginal of the state is the same in both regimes; the lagged cross-covariance flips sign
c, S = data["clean"].astype(float), data["S_s"]
rows = {}
for k in range(ds.K):
    m = S[:, :-5] == k
    a, b, b5 = c[:, :-5, 0], c[:, :-5, 1], c[:, 5:, 1]
    rows[f"regime {k}"] = {"mean s1": a[m].mean(), "var s1": a[m].var(), "var s2": b[m].var(), "E[s1(t) s2(t+5)]": (a * b5)[m].mean()}
pd.DataFrame(rows).round(3)

## 2. Feature learners, detectors, interventions

Every registered feature learner, with the same window and (for pair-trained learners) gap, plus an untrained `jepa_shared` control. `ts2vec` is the reference-faithful TS2Vec; `ts2vec_lite` is the earlier simplified version. `oracle_clean` applies the antisymmetric lagged cross-covariance (`lagcov`) to the clean 2-D state.

In [ ]:
SKIP = {"raw", "oracle", "jepa", "oracle_hidden"}
if D > 128:
    SKIP |= {"raw_cov", "raw_coskew", "raw_lagcov", "raw_bispec"}        # D^2 features: memory

STEPS = 20 if QUICK else 3000
TRAIN = dict(steps=STEPS, batch=128, workers=0 if QUICK else 1, eval_every=max(STEPS // 4, 5), warmup=min(1000, STEPS))

def learner(name, **extra):
    defaults = LEARNERS.defaults(name)
    p = dict(window=W)
    if "gap" in defaults:
        p["gap"] = GAP
    if "steps" in defaults:
        p.update(TRAIN)
    if name == "oracle_clean":
        p["feature"] = "lagcov"
    return make_learner(name, **{**p, **extra})

FEATURES = [learner(n) for n in LEARNERS.available() if n not in SKIP]
FEATURES.append(learner("jepa_shared", steps=0))   # control: untrained encoder
FIXED = [f for f in FEATURES if not f.trainable]
LEARNT = [f for f in FEATURES if f.trainable]
DETS = [dict(type=n) for n in DETECTORS.available()]

# counterfactual interventions (shared hidden trajectory / noise draws)
INTERVENTIONS = {"sensor": ({"sensor": 0}, {"sensor": 1}),
                 "dynamics": ({"main": 0, "sensor": 0}, {"main": 1, "sensor": 0})}
INV = dict(R=2, n=2000) if QUICK else dict(R=4, n=8000)
print(len(FIXED), "fixed and", len(LEARNT), "trained feature learners;", len(DETS), "detectors")

## 3. Fit, extract features, detect, measure sensitivity

Results go to `OUT` and `OUT_INV` after each learner. `oracle_clean` reads the clean state, so its sensitivity to the sensor is zero by construction and is not computed.

In [ ]:
bench = Benchmark(ds, detectors=DETS, out=RUNS, log=lambda *a: None)
results, inv = [], []
for f in FIXED + LEARNT:
    results.append(bench.run(f))
    views, starts = bench.features(f)
    row = dict(features_label=f.label(),
               decode_regime=decodability(views["features"], data["S_s"], starts, W, ds.K),
               decode_sensor=decodability(views["features"], data["S_sensor"], starts, W, 2))
    if f.input_key == "obs":
        row.update({f"sens_{k}": v for k, v in sensitivity(bench.fit(f), ds, INTERVENTIONS, **INV).items()})
    inv.append(row)
    pd.concat(results, ignore_index=True).to_csv(OUT, index=False); pd.DataFrame(inv).to_csv(OUT_INV, index=False)
    r = results[-1]; r = r[(r.view == "features") & r.metric.isin(["acc", "cp_ap"])]
    print(f"{f.label():<45}", "  ".join(f"{m} max {r[r.metric == m].value.max():.2f}" for m in ("acc", "cp_ap")),
          "  ".join(f"{k} {v:.2f}" for k, v in row.items() if k != "features_label"))
res, inv = pd.concat(results, ignore_index=True), pd.DataFrame(inv)

In [ ]:
for f in LEARNT:
    viz.plot_training(bench.run_dir(f), title=f.label())

## 4. Results

In [ ]:
viz.plot_results(res, metrics=("acc", "cp_ap", "cp_f1"))

In [ ]:
f_ = res[(res.view == "features") & (res.detector != "probe")]
acc = f_[f_.metric == "acc"]
best = pd.DataFrame({
    "acc static (gmm/hmm/temporal_excess)": acc[acc.detector.isin(["gmm", "hmm", "temporal_excess"])].groupby("features_label").value.max(),
    "acc switching dynamics (arhmm)": acc[acc.detector == "arhmm"].groupby("features_label").value.max(),
    "cp_f1 (best)": f_[f_.metric == "cp_f1"].groupby("features_label").value.max(),
}).join(inv.set_index("features_label"))
best.sort_values("acc switching dynamics (arhmm)", ascending=False).round(3)

**Sensitivity map.** Each point is a feature learner. The desired corner is top-left: large distance under a change of dynamics, small under a change of sensor condition.

In [ ]:
s = best.dropna(subset=["sens_sensor", "sens_dynamics"])
fig, ax = plt.subplots(figsize=(6.5, 5))
for name, r in s.iterrows():
    trained = any(name == f.label() for f in LEARNT)
    ax.scatter(r.sens_sensor, r.sens_dynamics, c="C3" if trained else "0.5", s=25)
    ax.annotate(name.split("(")[0] + (" (untrained)" if "steps=0" in name else ""), (r.sens_sensor, r.sens_dynamics), fontsize=6)
lim = [0, max(1.2, s[["sens_sensor", "sens_dynamics"]].max().max() * 1.05)]
ax.plot(lim, lim, "k:", lw=0.6); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel("sensitivity to the sensor condition"); ax.set_ylabel("sensitivity to the dynamics")
ax.set_title(f"D = {D}: red = trained, grey = fixed features", fontsize=8); plt.show()

## 5. Across observation dimensions

Every D (same rendering) whose results are on disk: the best static and switching-dynamics accuracies and the two sensitivities, per learner.

In [ ]:
rows = []
for p in sorted(glob.glob(f"cpbench_results/highdim_D*_{KIND}.csv")):
    d_ = int(re.search(r"_D(\d+)_", p).group(1)); r = pd.read_csv(p); r = r[(r.view == "features") & (r.metric == "acc")]
    iv = pd.read_csv(p.replace(".csv", "_invariance.csv")).set_index("features_label")
    for name, g in r.groupby("features_label"):
        rows.append(dict(D=d_, learner=name.split("(")[0] + (" (untrained)" if "steps=0" in name else ""),
                         static=g[g.detector.isin(["gmm", "hmm", "temporal_excess"])].value.max(), arhmm=g[g.detector == "arhmm"].value.max(),
                         **{k: iv.loc[name, k] for k in ("sens_sensor", "sens_dynamics") if name in iv.index and k in iv}))
sweep = pd.DataFrame(rows, columns=["D", "learner", "static", "arhmm", "sens_sensor", "sens_dynamics"])
if sweep.D.nunique() > 1:
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
    for ax, m in zip(axes, ("arhmm", "static", "sens_sensor")):
        t = sweep.pivot_table(index="D", columns="learner", values=m)
        t.plot(ax=ax, marker="o", lw=1, legend=ax is axes[0], logx=True); ax.set_title(m, fontsize=8)
    axes[0].legend(fontsize=5, ncol=2); plt.tight_layout(); plt.show()
if len(sweep):
    display(sweep.pivot_table(index="learner", columns="D", values=["arhmm", "static"]).round(3))
else:
    print("no full-run results on disk yet")